In [0]:
file_path = "/Volumes/bronze_raw_medical_data/azure_blob_storage/raw_medical_data/raw_patients.csv"

patients_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

In [0]:
patients_df = patients_df.withColumnsRenamed({
    'Id': 'patient_id',
    'BIRTHDATE__': 'birth_date',
    'DEATH_DATE': 'death_date',
    'PREFIX': 'prefix',
    'FIRST': 'first_name',
    'LAST': 'last_name',
    'SUFFIX': 'suffix',
    'MAIDEN': 'maiden',
    'race': 'race',
    'ETHNICITY': 'ethnicity',
    'gender': 'gender',
    'ADDRESS': 'address',
    'STATE': 'state',
    'COUNTY': 'county',
    'LAT': 'lat',
    'LON': 'lon'
})

In [0]:
from pyspark.sql.functions import col,try_to_date, to_date, coalesce, trim, date_format

patients_df = patients_df.withColumn('birth_date', trim(col("birth_date")))
patients_df = patients_df.withColumn('birth_date',
    coalesce(
        try_to_date(col("birth_date"), "yyyy-MM-dd"),
        try_to_date(col("birth_date"), "dd-MM-yyyy"),
        try_to_date(col("birth_date"), "MM-dd-yyyy")
    )
)
patients_df = patients_df.withColumn('birth_date',
    to_date(date_format(col("birth_date"), "yyyy-MM-dd"),"yyyy-MM-dd")
)


In [0]:
patients_df.write.format('delta').option('mergeSchema','true').mode('overwrite').saveAsTable('silver_medical_data.azure_blob_storage.patients')